<a href="https://colab.research.google.com/github/2403a54128/NLP/blob/main/nlp_assignment_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [1]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import numpy as np

# load elmo model

In [2]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")


# Dataset

In [3]:
sentences = ["Win money now", "Claim your prize", "Hello how are you",
             "Let's meet tomorrow", "Free lottery ticket", "Are you coming today"]

labels = [1, 1, 0, 0, 1, 0]

# Generate Embeddings

In [4]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']
print(embeddings.shape)

(6, 4, 1024)


# Inspect Word Embedding ("you")

In [5]:
# Convert sentences into tokens
tokenized = [sentence.split() for sentence in sentences]

# Generate ELMo embeddings (word-level)
elmo_outputs = elmo.signatures['default'](tf.constant(sentences))
embeddings = elmo_outputs['elmo']   # shape: (batch, words, 1024)

# Find index of "you" in sentences
idx1 = tokenized[2].index("you")   # "Hello how are you"
idx2 = tokenized[5].index("you")   # "Are you coming today"

# Extract embeddings
you_emb_1 = embeddings[2][idx1]
you_emb_2 = embeddings[5][idx2]

# Print first 10 values
print("First 10 values (Sentence 1):", you_emb_1[:10])
print("First 10 values (Sentence 2):", you_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[ 0.06630716  0.17725685 -0.03920957  0.24461518  0.47696626  0.6506039
 -0.3352741   0.5510186  -0.19608769  0.6226975 ], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.29958296 -0.0672917  -0.19454014 -0.04988772  0.08402111  0.49749476
 -0.0427475   0.7923057  -0.3325859   0.32552254], shape=(10,), dtype=float32)


# Compare Context (Cosine Similarity)

In [7]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(you_emb_1.numpy(), you_emb_1.numpy())

print("Similarity between 'YOU' meanings:", sim)

Similarity between 'YOU' meanings: 1.0


# Task - II

# BERT Model Implementation

In [8]:
sentences = [
    "The bat is flying",
    "He hit the ball with a bat"
]


In [9]:
# Preprocessing model
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# BERT encoder
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

# Preprocess Input

In [10]:
inputs = preprocess(sentences)
print(inputs.keys())

dict_keys(['input_type_ids', 'input_mask', 'input_word_ids'])


# Generate Embeddings

In [11]:
outputs = bert_model(inputs)

# Word-level embeddings
embeddings = outputs['sequence_output']

print("Embedding shape:", embeddings.shape)

Embedding shape: (2, 128, 768)


# Get Tokens

In [12]:
# Use tokenizer from preprocess model
tokenizer = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# Tokenized words (for visualization)
tokens = preprocess(sentences)['input_word_ids']

print(tokens)

tf.Tensor(
[[ 101 1996 7151 2003 3909  102    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]
 [ 101 2002 2718 1996 3608 2007 1037 7151  102    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    

# Extract bat Embeddings

In [13]:
bat_emb_1 = embeddings[0][2]  # "bat" in first sentence
bat_emb_2 = embeddings[1][7]  # "bat" in second sentence
print("First 10 values (Sentence 1):", bat_emb_1[:10])
print("First 10 values (Sentence 2):", bat_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-6.0759485e-05  2.6044574e-01  1.2677923e-01 -2.8091615e-01
  1.3291722e-02 -4.8430106e-01  7.0899105e-01  7.3061728e-01
  2.0883086e-01 -7.8647327e-01], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.06056874 -0.9072487  -0.06744917 -0.27453542 -0.46633738  0.04165877
  0.23526224  0.35728985  0.9171287  -0.6723436 ], shape=(10,), dtype=float32)


# Cosine Similarity

In [14]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bat_emb_1.numpy(), bat_emb_2.numpy())

print("Similarity between 'bat' meanings:", sim)

Similarity between 'bat' meanings: 0.43427736


# ELMo + Naive Bayes


# import libraries

In [15]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# corpus

In [16]:
sentences = [
    "Congratulations you won a free iPhone",
    "Claim your cash prize now",
    "Hey how are you doing",
    "Let's have lunch tomorrow",
    "Limited offer buy now",
    "Are you available for meeting",
    "Win money instantly click here",
    "Good morning have a nice day"
]

labels = [1, 1, 0, 0, 1, 0, 1, 0]   # 1 = spam, 0 = normal

# Load ELMo Model

In [17]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

# Generate ELMo Embeddings

In [18]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']

print("Shape:", embeddings.shape)

Shape: (8, 6, 1024)


# Convert to Sentence Embeddings

In [19]:
sentence_embeddings = tf.reduce_mean(embeddings, axis=1)

X = sentence_embeddings.numpy()
y = np.array(labels)

print("Feature shape:", X.shape)

Feature shape: (8, 1024)


# Train-Test Split

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# Train Model

In [21]:
model = GaussianNB()
model.fit(X_train, y_train)

GaussianNB()

# Model Testing

In [22]:
y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("Actual:", y_test)

Predictions: [1 1]
Actual: [1 0]


# Model Evaluation

In [23]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.5


# Prediction on New Text

In [26]:
new_sentence = ["Congratulations! You won a free ice-cream"]

new_emb = elmo.signatures['default'](tf.constant(new_sentence))['elmo']
new_emb = tf.reduce_mean(new_emb, axis=1)

prediction = model.predict(new_emb.numpy())

print("Prediction:", "Spam" if prediction[0] == 1 else "Not Spam")

Prediction: Spam


# Text Classification using BERT+NB

In [27]:
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

In [28]:
bert_inputs = preprocess(sentences)

In [29]:
bert_outputs = bert_model(bert_inputs)

# Use sentence embedding ([CLS])
bert_features = bert_outputs['pooled_output'].numpy()

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    bert_features, labels, test_size=0.2, random_state=42
)

nb_bert = GaussianNB()
nb_bert.fit(X_train, y_train)

y_pred_bert = nb_bert.predict(X_test)
acc_bert = accuracy_score(y_test, y_pred_bert)

print("BERT Accuracy:", acc_bert)

BERT Accuracy: 0.5
